# Haplotype-based selection: iHS and XP-EHH

**Purpose.** The frequency-based statistics ask whether allele *frequencies* look unusual.
Haplotype methods ask something different and more recent: when a variant sweeps up quickly,
it drags its whole haplotype with it, so carriers of the selected allele share a long,
**unbroken, homozygous** stretch of chromosome that recombination has not yet had time to
break up. Measuring that extended haplotype homozygosity detects sweeps that are still in
progress, which frequency methods can miss.

**What you will do**
 - look at the phased VCF and the genetic map, and see why both are needed
 - run `selscan --ihs` to compute the **integrated haplotype score** from extended
   haplotype homozygosity
 - normalise iHS within allele-frequency bins, and understand why that is necessary
 - run `selscan --xpehh` to compare two populations directly, using West Africans as the
   reference
 - check whether the known causal variant stands out

**The data.** Three populations from the **1000 Genomes Project**, phased genotypes around
the **lactase (LCT)** region:

| Code | Population | n |
|---|---|---|
| CEU | Utah residents with northern/western European ancestry | 41 |
| YRI | Yoruba in Ibadan, Nigeria | 48 |
| CHB | Han Chinese in Beijing | 48 |

**Called genotypes, and they must be phased** — haplotype methods read along a chromosome,
so unphased genotypes are useless to them. A **genetic map in centimorgans** is supplied as
well: extended haplotype homozygosity has to be measured against recombination distance, not
physical distance, or regions of low recombination look selected everywhere.

The LCT region is a **positive control**: lactase persistence is one of the strongest and
most recent sweeps in Europeans, so CEU should show a signal and YRI should not.

**Note.** The `selscan` runs take several minutes each, so the commands are shown and the
precomputed output is supplied.

**Before this** do [SFS, Fst and PBS](sfs_fst_pbs_human.ipynb) and
[the genome-wide PBS scan](selection_pbs_scan_human.ipynb), which use the frequency-based
statistics this exercise is the counterpart to.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
ThePath=/course/data/current_data/selection/haplotype

# the selscan program folder
SS=$ThePath/prog/selscan-linux-1.3.0

# the phased VCF files, one per population
ceuVCF=$ThePath/ceuLCT.recode.vcf
yriVCF=$ThePath/yriLCT.recode.vcf
chbVCF=$ThePath/chbLCT.recode.vcf

# genetic map, positions in centimorgans
MAP=$ThePath/geneticV2.map

# where you will do the exercise
WORK_DIR=$HOME/selection_haplotype_human
mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.selection_haplotype_workdir
cd $WORK_DIR

echo --- data ---
ls $ThePath
echo; echo --- precomputed selscan output ---
ls $ThePath/run | head

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.selection_haplotype_workdir"))[1]
setwd(work_d)
getwd()

And let's quickly look at a line in the vcf file to see how the fact that the data is indicated:

In [ ]:
tail -n 1 $ThePath/ceuLCT.recode.vcf

# Exercise II: Haplotype-based methods

## Integrated Haplotype Score (iHS)

Lets now see if the haplotype homozygosity does a better job than the frequency-based methods.

Run iHS using the command:

In [ ]:
#$SS/selscan --ihs --vcf $ceuVCF --pmap --out ceuLCT --threads 8

**Questions**
 - The `selscan --ihs` command is commented out because it takes minutes. Reading it: which two inputs does it need besides the VCF?
 - Why does iHS need a genetic map rather than physical positions?

The analysis takes a couple of minutes so instead we will work with an already pre-ran set of results. You can copy it here:

In [ ]:
ln -s $ThePath/run/ceuLCT.ihs* .

**Question**
 - The output columns are locus, position, frequency of the '1' allele, ihh1, ihh0 and unstandardised iHS. What does the ratio of ihh1 to ihh0 measure?

The output columns are:

```
<locusID> <physicalPos> <'1' freq> <ihh1> <ihh0> <unstandardized iHS>
```

These statistics will be affected by the frequency of the SNPs therefore we have to normalize in frequency bins. The default in 100 bins.

In [ ]:
$SS/norm --ihs --files ceuLCT.ihs.out --bins 20

**Questions**
 - Why must iHS be normalised **within allele-frequency bins** rather than globally?
 - The exercise says 20 bins is too many here. What goes wrong when a bin has too few SNPs?

The number of bins is too high for this data set since we do not have enough SNPs for each bin of allele frequencies. Therefore, redo the analysis where you reduce the number of bins to 20 with the `--bins 20` option.

Lets plot the results in `R`:

In [ ]:
r <- read.table("ceuLCT.ihs.out.20bins.norm",as.is=T,head=F)
names(r) <- c("locusID", "physicalPos","freq","ihh1","ihh0","unstandardizediHS")
r[which.max(r$ihh1/r$ihh0),]
causalSNP <- 136608646
#plot without frequency standardization
plot(r$physicalPos,r$unstandardizediHS);

## with standardiztion IHS=ihh0/ihh1
r$iHS<-log(r$ihh1/r$ihh0)
plot(r$physicalPos,r$iHS,ylab="iHs");
abline(v=causalSNP,col="red")

## causal SNP test statistics vs. rest of region
(causalSite<-r[which(r$physicalPos==causalSNP),])
hist(r$iHS)
abline(v=causalSite$iHS,col="red")


**Questions**
 - Does the causal site stand out from its neighbours?
 - iHS compares the two alleles at the *same* site. What kind of sweep would that fail to detect?

## Cross-population EHH (XP-EHH)

Lets try to use West Africans (YRI) to normalise the iHS with XP-EHH:

In [ ]:
#$SS/selscan --xpehh --vcf $ceuVCF --pmap --vcf-ref $yriVCF --out ceuLCT --threads 8

This may take up to 10 minutes so feel free to copy the results instead:

In [ ]:
ln -s $ThePath/run/ceuLCT.xpehh* .

**Question**
 - XP-EHH compares CEU against YRI rather than the two alleles within CEU. What can it detect that iHS cannot?

We also have to normalize these results:

In [ ]:
$SS/norm --xpehh --files ceuLCT.xpehh.out

And once more, you can plot the results in `R`:

In [ ]:
r<-read.table("ceuLCT.xpehh.out.norm",head=T,as.is=T,row.names=NULL)
causalSNP <- 136608646
(causalSite<-r[which(r$pos==causalSNP),])                 
plot(r$pos,r$normxpehh)
abline(v=causalSNP,col="red")

#print the site with maximum statistic 
r[which.max(r$normxpehh),]

#plot the distribution
hist(r$normxpehh)
abline(v=causalSite$normxpehh,col="red")

#get the quantile
mean(causalSite$normxpehh>r$normxpehh)

**Questions**
 - Is the causal site more convincing under XP-EHH than under iHS?
 - YRI is the reference here. What would change if you used CHB instead, and would you expect the LCT signal to survive?

**- Are you more convinced that the site is under selection?**

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/selection/quiz/haplotype_selection.json")
